# FLAML AutoML CSV Upload Demo\n\nUpload a CSV file, select the target column, train an AutoML model with FLAML, then run a sample prediction.

In [ ]:
import io\nimport numpy as np\nimport pandas as pd\nfrom flaml import AutoML\nimport ipywidgets as widgets\nfrom IPython.display import display

In [ ]:
uploader = widgets.FileUpload(accept='.csv', multiple=False, description='Upload CSV')\ndisplay(uploader)

In [ ]:
if not uploader.value:\n    raise ValueError('Please upload a CSV file first, then re-run this cell.')\n\nuploaded_file = next(iter(uploader.value.values()))\ncontent = uploaded_file['content']\ndf = pd.read_csv(io.BytesIO(content))\nprint(f'Data shape: {df.shape}')\ndisplay(df.head())

In [ ]:
target_selector = widgets.Dropdown(options=df.columns.tolist(), description='Target:')\ndisplay(target_selector)

In [ ]:
target_col = target_selector.value\nX = df.drop(columns=[target_col]).copy()\ny = df[target_col].copy()\n\nfor col in X.select_dtypes(include=['object', 'category']).columns:\n    X[col] = X[col].astype('category').cat.codes\n\nautoml = AutoML()\nautoml_settings = {\n    'time_budget': 30,\n    'metric': 'accuracy',\n    'task': 'classification',\n    'log_file_name': 'flaml.log',\n}\nautoml.fit(X_train=X, y_train=y, **automl_settings)\n\nprint('Best learner:', automl.best_estimator)\nprint('Best config:', automl.best_config)

In [ ]:
example = X.iloc[[0]].copy()\nprediction = automl.predict(example)\nprint('Example input row:')\ndisplay(example)\nprint('Predicted class:', prediction[0])